# LLM Process

A notebook used to simplify (ie. remove columns from & restructure) the dataset for LLM processing

In [2]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt 

## Setup

In [3]:
data_dir = "../raw"
filename = data_dir + "/merged_2021_2025.csv"
raw_df = df = pd.read_csv(filename, index_col=0, header=[0, 1], low_memory=False)

In [4]:
flat = raw_df.copy()
flat.columns = [f"{a.replace(' ', "_")}_{b.replace(' ', "_")}" for a, b in flat.columns]
flat.head()

,Time_Date,Time_Local_Time_Of_Day,Place_Locale_Reference,Place_State_Reference,Place_Relative_Position.Angle.Radial,Place_Relative_Position.Distance.Nautical_Miles,Place_Altitude.AGL.Single_Value,Place_Altitude.MSL.Single_Value,Place_Latitude_/_Longitude_(UAS),Environment_Flight_Conditions,...,Events_Detector,Events_When_Detected,Events_Result,Assessments_Contributing_Factors_/_Situations,Assessments_Primary_Problem,Report_1_Narrative,Report_1_Callback,Report_2_Narrative,Report_2_Callback,Report_1_Synopsis
2260174,202507,1201-1800,BWI.Airport,MD,NaN,NaN,NaN,900.0,NaN,VMC,...,Person Air Traffic Control,In-flight,General None Reported / Taken,Procedure; Human Factors; ATC Equipment / Nav ...,Ambiguous,Aircraft vectored in within 1NM to final appro...,NaN,NaN,NaN,Air carrier Captain reported a low altitude al...
2260249,202507,0601-1200,ZZZ.Airport,US,NaN,NaN,NaN,NaN,NaN,NaN,...,Automation Aircraft Other Automation; Person F...,In-flight,Flight Crew Executed Go Around / Missed Approa...,Human Factors; Aircraft,Ambiguous,While on short final we received a glideslope ...,NaN,NaN,NaN,Air carrier pilot reported receiving an aircra...
2260370,202507,1801-2400,ZZZ.ARTCC,US,NaN,NaN,NaN,35000.0,NaN,VMC,...,Person Flight Crew,In-flight,Air Traffic Control Issued New Clearance; Air ...,Aircraft,Aircraft,Flying at cruise; FL350; the FO was the PF and...,NaN,At cruise; FL350; during level-off; the Captai...,NaN,B737 flight crew reported observing a slowing ...
2261277,202507,1801-2400,ZZZ.Airport,US,NaN,2.6,160.0,NaN,NaN,NaN,...,Person Flight Crew,In-flight,General None Reported / Taken,Human Factors,Human Factors,On Day 0 around XA:30; I forgot to get LAANC a...,Reporter stated TRUST certificate was obtained...,NaN,NaN,Recreational / Hobbyist UAS pilot reported fly...
2261317,202507,NaN,ZZZ.Airport,US,NaN,NaN,NaN,9000.0,NaN,VMC,...,Automation Aircraft Other Automation; Person F...,In-flight,Flight Crew Became Reoriented; Flight Crew Reg...,Weather; Human Factors; Procedure,Procedure,Divert into ZZZ from ZZZ1. FO flying. Vectors ...,NaN,Extremely strong winds blew us off the LOC whi...,NaN,B737 flight crew reported the pilot flying in ...


In [27]:
# print(flat.iloc[0].to_json())
tt = flat[flat.Assessments_Primary_Problem == 'Human Factors']
with pd.option_context(
    "display.max_rows", None,
):  # more options can be specified also
    print(

    tt[[
        "Assessments_Primary_Problem",
        "Person_1_Human_Factors",
        # "Person_2_Human_Factors",
        # "",
    ]].map(lambda x : str(set(x.lower().split("; "))) if isinstance(x, str) else "NaN").value_counts()
    )

Assessments_Primary_Problem  Person_1_Human_Factors                                                                                                                                                      
{'human factors'}            {'situational awareness'}                                                                                                                                                       1043
                             {'situational awareness', 'communication breakdown'}                                                                                                                             819
                             {'communication breakdown'}                                                                                                                                                      812
                             NaN                                                                                                                                        

In [6]:
narrative_only = flat[
    ["Assessments_Primary_Problem", "Report_1_Narrative", "Report_2_Narrative"]
]

### Cleaning Target & De-duplication
Here we examine if the target is NaN or dataset has any duplicates. Both Report_1_Narrative & Report_2_Narrative have duplicates (which shouldn't be the case).

In [15]:
narrative_only.info()

<class 'pandas.DataFrame'>
Index: 25526 entries, 2260174 to 2315041
Data columns (total 3 columns):
 #   Column                       Non-Null Count  Dtype
---  ------                       --------------  -----
 0   Assessments_Primary_Problem  25089 non-null  str  
 1   Report_1_Narrative           25526 non-null  str  
 2   Report_2_Narrative           5389 non-null   str  
dtypes: str(3)
memory usage: 797.7 KB


In [16]:
narrative_only.describe()

,Assessments_Primary_Problem,Report_1_Narrative,Report_2_Narrative
count,25089,25526,5389
unique,97,25522,5224
top,Human Factors,The UAS lost the telemetry link 1;300 feet fro...,[Report narrative contained no additional info...
freq,8370,3,125


In [17]:
# Identify Report 1 Narrative that are duplicated

r1_dupes = narrative_only[
    narrative_only.Report_1_Narrative.duplicated(keep=False)
]
r1_dupes.dropna(how="all")

,Assessments_Primary_Problem,Report_1_Narrative,Report_2_Narrative
2291690,Aircraft,On approach into ZZZ runway XXR; landing gear ...,After an otherwise uneventful flight; while on...
2014961,Aircraft,The UAS lost the telemetry link 1;300 feet fro...,NaN
2014962,Aircraft,The UAS lost the telemetry link 1;300 feet fro...,NaN
2014963,Aircraft,The UAS lost the telemetry link 1;300 feet fro...,NaN
2233790,Human Factors,In attempt to do my short field landing coming...,NaN
2242665,Ambiguous,In attempt to do my short field landing coming...,NaN
2303286,Aircraft,On approach into ZZZ runway XXR; landing gear ...,After an otherwise uneventful flight; while on...


In [25]:
r2_only = narrative_only.copy()
r2_dupes = r2_only[r2_only.Report_2_Narrative.duplicated(keep=False) & pd.notna(r2_only.Report_2_Narrative)]
r2_dupes.dropna(how="all")
display(r2_dupes.Report_2_Narrative.value_counts())
print("Sentinel values have lots of duplicates.")


Report_2_Narrative
[Report narrative contained no additional information.]    125
[Narrative contained no additional information.]            24
[Report narrative contained no additional information]      10
[Report narrative contained no additional information].      7
[Narrative added no additional information.]                 2
[Narrative contained no additional information]              2
[Report contained no additional information.]                2
Name: count, dtype: int64

Sentinel values have lots of duplicates.


In [26]:
# Find all occurrences of sentinel value string to ensure we don't delete actual data
# r2_no_sentinel = []
sentinel = "contained no additional information"
print("R1 Narrative w/ sentinels")
display(narrative_only[
    narrative_only.Report_1_Narrative.str.contains(sentinel)
])
print("R2 Narrative w/ sentinels")
display(narrative_only[narrative_only.Report_2_Narrative.str.contains(sentinel)][
    ["Report_2_Narrative"]
].value_counts())
print("So we can safely replace sentinel string w/ NaN & not delete real info.")

R1 Narrative w/ sentinels


,Assessments_Primary_Problem,Report_1_Narrative,Report_2_Narrative


R2 Narrative w/ sentinels


Report_2_Narrative                                     
[Report narrative contained no additional information.]    125
[Narrative contained no additional information.]            24
[Report narrative contained no additional information]      10
[Report narrative contained no additional information].      7
[Narrative contained no additional information]              2
[Report contained no additional information.]                2
Second narrative contained no additional information.        1
Name: count, dtype: int64

So we can safely replace sentinel string w/ NaN & not delete real info.


In [27]:
no_nan_r2_dupes = r2_dupes[~r2_dupes.Report_2_Narrative.str.contains(sentinel)]
no_nan_r2_dupes.sort_values(by="Report_2_Narrative").index


Index([1811991, 1851270], dtype='int64')

In [21]:
# Searching these indices on the DB, they indeed do NOT have a primary problem assigned.
prob_null = narrative_only[pd.isna(narrative_only.Assessments_Primary_Problem)]
prob_null

,Assessments_Primary_Problem,Report_1_Narrative,Report_2_Narrative
1780552,NaN,ZZZ1 went ATC zero due to a COVID cleaning aro...,NaN
1781139,NaN,At touchdown; aircraft started to pull right a...,NaN
1782360,NaN,I was the PM (Pilot Monitoring) and the CA (Ca...,NaN
1782374,NaN,On approach to Runway XX at ZZZ. Pilot flying ...,NaN
1785122,NaN,Was working these two sectors combined. It nee...,NaN
...,...,...,...
2312691,NaN,On Day 0; between XA30z -XC00z; operating as P...,NaN
2314212,NaN,Exiting the runway on EWR Runway 22R at night ...,NaN
2314249,NaN,On Day 0 we were given an IFR clearance from S...,On departure after climbing to 7.000 on the he...
2314640,NaN,Sent a CPDLC route change for direct IOW (Iowa...,While in Cruise in contact with Cleveland Cent...


After referencing the DB, we can: 
1) Drop all items where the primary problem is NaN (since there is no label available)
2) Deduplicate & keep the first Report_1_Narrative (indeed, the narratives are exactly the same and in ONE case -- ACN 2233790/2242665 -- they have different primary problem labels)
3) Replace sentinel values (eg. "contained not additional info") with NaN
4) Apply Keep-first-override to all the duplicate Report_2_Narratives (see README) & null Report_1_Narratives

In [28]:
sentinel_substring = "contained no additional information"
sentinel_substring_2 = "narrative added no additional information"

def convert_sentinel(s:str):
    if not isinstance(s, str) or sentinel_substring in s.lower() or sentinel_substring_2 in s.lower():
        return np.nan
    
    return s


filtered = narrative_only[pd.notna(narrative_only.Assessments_Primary_Problem)].copy()
filtered = filtered.drop_duplicates(["Report_1_Narrative"], keep="first")
filtered["Report_2_Narrative"] = filtered.Report_2_Narrative.apply(convert_sentinel)

filtered.describe()

,Assessments_Primary_Problem,Report_1_Narrative,Report_2_Narrative
count,25085,25085,5125
unique,97,25085,5125
top,Human Factors,Aircraft vectored in within 1NM to final appro...,At cruise; FL350; during level-off; the Captai...
freq,8370,1,1


## Normalize Target
Now we make sure the Primary Problem is just one of the major categories (not subcategories).


In [29]:
category_counts = filtered.Assessments_Primary_Problem.value_counts()
print(category_counts.to_string())

Assessments_Primary_Problem
Human Factors                                                           8370
Aircraft                                                                8193
Procedure                                                               2410
Ambiguous                                                               1983
Weather                                                                  823
Airport                                                                  610
Environment - Non Weather Related                                        594
Chart Or Publication                                                     336
Airspace Structure                                                       322
ATC Equipment / Nav Facility / Buildings                                 316
Software and Automation                                                  169
Company Policy                                                           132
Equipment / Tooling                             

In [30]:
s = 0
for c, r in zip(category_counts.index, category_counts):
    li = c.replace(" ", "").split(";") #bc sometimes we have things like 'aircraft; aircraft'
    if len(set(li)) > 1:
        s+= r
    
print(s, "records are a multi-class target")
print(round(s / category_counts.sum(), 4), "of all records")

416 records are a multi-class target
0.0166 of all records


In [31]:
def setify(x:str):
    return set(x.lower().replace(" ", "").split(";"))


processed = filtered.copy()
processed["setify"] = processed.Assessments_Primary_Problem.apply(setify)
processed['is_multi'] = processed.setify.apply(lambda x : len(x) > 1)
processed = processed[~processed.is_multi]
processed["Assessments_Primary_Problem"] = processed.setify.apply(lambda x : list(x)[0])

display(processed.Assessments_Primary_Problem.value_counts())
processed.drop(['setify', 'is_multi'], axis=1, inplace=True)
processed.head()

Assessments_Primary_Problem
humanfactors                              8427
aircraft                                  8272
procedure                                 2418
ambiguous                                 1989
weather                                    823
airport                                    610
environment-nonweatherrelated              595
chartorpublication                         336
airspacestructure                          322
atcequipment/navfacility/buildings         316
softwareandautomation                      169
companypolicy                              132
equipment/tooling                           96
mel                                         59
incorrect/notinstalled/unavailablepart      48
staffing                                    27
manuals                                     25
logbookentry                                 5
Name: count, dtype: int64

,Assessments_Primary_Problem,Report_1_Narrative,Report_2_Narrative
2260174,ambiguous,Aircraft vectored in within 1NM to final appro...,NaN
2260249,ambiguous,While on short final we received a glideslope ...,NaN
2260370,aircraft,Flying at cruise; FL350; the FO was the PF and...,At cruise; FL350; during level-off; the Captai...
2261277,humanfactors,On Day 0 around XA:30; I forgot to get LAANC a...,NaN
2261317,procedure,Divert into ZZZ from ZZZ1. FO flying. Vectors ...,Extremely strong winds blew us off the LOC whi...


In [32]:
processed.to_csv("narratives.csv")

In [ ]:
# Count tokens, roughly
processed.Report_1_Narrative.apply(lambda x: x.count(" ")).sum()

np.int64(4843308)